<a href="https://colab.research.google.com/github/antoniaacuna/TESIS-ACUNA/blob/main/TESIS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
video = files.upload()


# **Elección de un frame de referencia**

1.   Correr la celda. Va a aparecer el frame del video (segundo 10) con 4 controles deslizantes: X inicio, Y inicio, Ancho, Alto.


2. Mover los sliders hasta que el rectángulo verde quede así:
- La aplysia completa adentro del cuadro (cuerpo y "patas"/parápodos si están extendidos).

- Un poco de margen extra alrededor, especialmente en el eje X (izquierda y derecha) — el animal se estira y se mueve al caminar, así que conviene dejar más aire de ese lado que arriba/abajo.

- El borde inferior del cuadro apoyado justo sobre el piso del tanque (dejar el menor espacio entre el animal y el piso).




![descripción](https://drive.google.com/uc?export=view&id=1GYVNwo0KbKmP-NoplLFQbhFAjh7YIkwx)

In [ ]:
#Paso 1: Elección frame de referencia
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import ipywidgets as widgets
from IPython.display import display

nombre_video = list(video.keys())[0]
cap = cv2.VideoCapture(nombre_video)
cap.set(cv2.CAP_PROP_POS_MSEC, 10000)
ret, frame_roi = cap.read()
cap.release()

frame_rgb = cv2.cvtColor(frame_roi, cv2.COLOR_BGR2RGB)
alto, ancho = frame_rgb.shape[:2]

MARGEN_Y_PX = 18
MARGEN_X_PORCENTAJE = 0.22

roi_coords = {"x": 0, "y": 0, "w": 0, "h": 0}
roi_coords_final = {"x": 0, "y": 0, "w": 0, "h": 0}

sx = widgets.IntSlider(min=0, max=ancho-1, value=200, description='X inicio')
sy = widgets.IntSlider(min=0, max=alto-1, value=200, description='Y inicio')
sw = widgets.IntSlider(min=10, max=ancho, value=300, description='Ancho')
sh = widgets.IntSlider(min=10, max=alto, value=150, description='Alto')

out = widgets.Output()

def actualizar(x, y, w, h):
    roi_coords["x"] = x
    roi_coords["y"] = y
    roi_coords["w"] = w
    roi_coords["h"] = h

    margen_x = int(w * MARGEN_X_PORCENTAJE)
    x_final = max(0, x - margen_x)
    w_final = min(ancho, x + w + margen_x) - x_final
    y_final = max(0, y - MARGEN_Y_PX)
    h_final = min(alto, y + h + MARGEN_Y_PX) - y_final

    roi_coords_final["x"] = x_final
    roi_coords_final["y"] = y_final
    roi_coords_final["w"] = w_final
    roi_coords_final["h"] = h_final

    with out:
        out.clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(13, 6))
        ax.imshow(frame_rgb)

        rect = patches.Rectangle((x, y), w, h, linewidth=2, edgecolor='lime', facecolor='none')
        ax.add_patch(rect)

        rect_final = patches.Rectangle((x_final, y_final), w_final, h_final,
                                        linewidth=2, edgecolor='yellow', linestyle='--', facecolor='none')
        ax.add_patch(rect_final)

        ax.set_title(f"Verde = Ajuste manual  |  Amarillo punteado = ROI final (con margen)\n"
                     f"Final: x={x_final}, y={y_final}, w={w_final}, h={h_final}", fontsize=11)
        plt.tight_layout()
        plt.show()

ui = widgets.VBox([sx, sy, sw, sh])
widgets.interactive_output(actualizar, {'x': sx, 'y': sy, 'w': sw, 'h': sh})
display(ui, out)
actualizar(sx.value, sy.value, sw.value, sh.value)


In [ ]:
#Uso SOLO en casos con 1 error que afecta el cálculo USAR POST COMPLETAR TODO EL CICLO
import cv2, base64
from google.colab import output
from IPython.display import display, HTML

nombre_video = list(video.keys())[0]
SEGUNDO_A_VER = 0

cap = cv2.VideoCapture(nombre_video)
fps = cap.get(cv2.CAP_PROP_FPS)
cap.set(cv2.CAP_PROP_POS_FRAMES, int(SEGUNDO_A_VER * fps))
ret, frame = cap.read()
cap.release()

_, buf = cv2.imencode('.png', frame)
b64 = base64.b64encode(buf).decode()

ZONA_MANUAL_RESULTADO = {}

def _guardar_zona(x1, y1, x2, y2):
    global ZONA_MANUAL_RESULTADO
    x, y = int(min(x1, x2)), int(min(y1, y2))
    w, h = int(abs(x2 - x1)), int(abs(y2 - y1))
    ZONA_MANUAL_RESULTADO = {"x": x, "y": y, "w": w, "h": h}
    print(f"Zona guardada (coordenadas reales de la imagen): x={x}, y={y}, w={w}, h={h}")

output.register_callback('notebook.guardar_zona', _guardar_zona)

html = f'''
<div>
  <canvas id="canvas_zona" style="cursor:crosshair;border:1px solid #333;max-width:100%;height:auto;"></canvas>
  <p>Hacer click y arrastrar sobre la imagen para marcar el rectángulo. Soltar el mouse para confirmar.</p>
</div>
<script>
var img = new Image();
img.onload = function() {{
  var canvas = document.getElementById('canvas_zona');
  canvas.width = img.width;
  canvas.height = img.height;
  var ctx = canvas.getContext('2d');
  ctx.drawImage(img, 0, 0);
  var startX, startY, drawing = false;

  function coordsReales(e) {{
    var r = canvas.getBoundingClientRect();
    var escalaX = canvas.width / r.width;
    var escalaY = canvas.height / r.height;
    return [(e.clientX - r.left) * escalaX, (e.clientY - r.top) * escalaY];
  }}

  canvas.onmousedown = function(e) {{
    var c = coordsReales(e);
    startX = c[0]; startY = c[1];
    drawing = true;
  }};
  canvas.onmousemove = function(e) {{
    if (!drawing) return;
    var c = coordsReales(e);
    var curX = c[0], curY = c[1];
    ctx.drawImage(img, 0, 0);
    ctx.strokeStyle = 'red';
    ctx.lineWidth = 2;
    ctx.strokeRect(startX, startY, curX - startX, curY - startY);
  }};
  canvas.onmouseup = function(e) {{
    drawing = false;
    var c = coordsReales(e);
    google.colab.kernel.invokeFunction('notebook.guardar_zona', [startX, startY, c[0], c[1]], {{}});
  }};
}};
img.src = "data:image/png;base64,{b64}";
</script>
'''
display(HTML(html))

In [ ]:
# CELDA 1
import cv2
import numpy as np

KERNEL_GAUSS = 7
KERNEL_VAR   = 25
V_MAX_TEXTURA = 70
FACTOR_UMBRAL = 1.05
N_MUESTRAS = 60
KERNEL_RELLENO = 15
AREA_MIN_ESTRUCTURA = 150
PISO_UMBRAL_ESTRUCTURA = 0.75

ZONA_MANUAL_RESULTADO = globals().get("ZONA_MANUAL_RESULTADO", {})

if ZONA_MANUAL_RESULTADO:
    ZONAS_MANUALES = [(ZONA_MANUAL_RESULTADO["x"], ZONA_MANUAL_RESULTADO["y"], ZONA_MANUAL_RESULTADO["w"], ZONA_MANUAL_RESULTADO["h"])]
else:
    ZONAS_MANUALES = []

GABOR_PARAMS = [
    {"freq": 1/12, "theta": 0},
    {"freq": 1/12, "theta": np.pi/4},
    {"freq": 1/16, "theta": 0},
    {"freq": 1/16, "theta": np.pi/4},
]
kernels = [cv2.getGaborKernel((31, 31), 4.0, p["theta"], 1.0/p["freq"], 0.5, 0, ktype=cv2.CV_32F)
           for p in GABOR_PARAMS]

def varianza_local(gris, kernel_size):
    f = gris.astype(np.float32)
    return np.clip(cv2.blur(f**2, (kernel_size, kernel_size)) - cv2.blur(f, (kernel_size, kernel_size))**2, 0, None)

def binarizar_animal_oscuro(gris, combinado, factor_umbral=FACTOR_UMBRAL):
    t_otsu, _ = cv2.threshold(combinado, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    t_ajustado = float(np.clip(t_otsu * factor_umbral, 0, 255))
    _, mask_a = cv2.threshold(combinado, t_ajustado, 255, cv2.THRESH_BINARY)
    _, mask_b = cv2.threshold(combinado, t_ajustado, 255, cv2.THRESH_BINARY_INV)
    brillo_a = gris[mask_a == 255].mean() if np.any(mask_a == 255) else 255
    brillo_b = gris[mask_b == 255].mean() if np.any(mask_b == 255) else 255
    return mask_a if brillo_a < brillo_b else mask_b

nombre_video = list(video.keys())[0]
cap = cv2.VideoCapture(nombre_video)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
ancho = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
alto  = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

indices_muestra = np.linspace(0, total_frames - 1, N_MUESTRAS, dtype=int)
acumulador = np.zeros((alto, ancho), dtype=np.float32)
frames_leidos = 0

for idx in indices_muestra:
    cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ret, frame = cap.read()
    if not ret:
        continue
    frames_leidos += 1

    v_original = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)[:, :, 2]
    suave  = cv2.GaussianBlur(frame, (KERNEL_GAUSS, KERNEL_GAUSS), 0)
    gris   = cv2.cvtColor(suave, cv2.COLOR_BGR2GRAY)
    gris   = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8)).apply(gris)
    gris_f = gris.astype(np.float32)

    suma = np.zeros_like(gris_f)
    for k in kernels:
        suma += np.abs(cv2.filter2D(gris_f, cv2.CV_32F, k))
    gabor_inv = cv2.bitwise_not(cv2.normalize(suma / len(kernels), None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8))
    var_norm  = cv2.normalize(varianza_local(gris, KERNEL_VAR), None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    combinado = cv2.addWeighted(gabor_inv, 0.6, var_norm, 0.4, 0)

    mask = binarizar_animal_oscuro(gris, combinado)
    mask_oscuro = (v_original < V_MAX_TEXTURA).astype(np.uint8) * 255
    mask = cv2.bitwise_and(mask, mask_oscuro)

    acumulador += (mask > 0).astype(np.float32)

cap.release()

frecuencia = acumulador / max(frames_leidos, 1)
frecuencia_u8 = (frecuencia * 255).astype(np.uint8)
t_otsu_frec, _ = cv2.threshold(frecuencia_u8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
umbral_relativo = max(t_otsu_frec / 255.0, PISO_UMBRAL_ESTRUCTURA)

mask_previa = (frecuencia >= umbral_relativo).astype(np.uint8) * 255

mask_cerrada = cv2.morphologyEx(
    mask_previa, cv2.MORPH_CLOSE,
    cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (KERNEL_RELLENO, KERNEL_RELLENO))
)

n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask_cerrada, connectivity=8)
mask_auto = np.zeros_like(mask_cerrada)
for i in range(1, n_labels):
    if stats[i, cv2.CC_STAT_AREA] >= AREA_MIN_ESTRUCTURA:
        mask_auto[labels == i] = 255

mask_manual = np.zeros_like(mask_auto)
for (zx, zy, zw, zh) in ZONAS_MANUALES:
    mask_manual[zy:zy+zh, zx:zx+zw] = 255

MASCARA_ESTRUCTURAS = cv2.bitwise_or(mask_auto, mask_manual)

pct = 100 * np.mean(MASCARA_ESTRUCTURAS == 255)
print(f"Umbral automático para este video: {umbral_relativo:.3f}")
print(f"Zonas manuales cargadas: {len(ZONAS_MANUALES)}")
print(f"Estructuras fijas totales (auto + manual): {pct:.1f}% del frame (frames muestreados: {frames_leidos})")

In [ ]:
#CELDA 2
import cv2
import numpy as np
import base64
import subprocess
import csv
from collections import deque
from IPython.display import HTML, display

if 'MASCARA_ESTRUCTURAS' not in globals():
    raise RuntimeError("Falta correr la CELDA 1 antes de esto")

KERNEL_GAUSS = 7
KERNEL_VAR   = 25
AREA_MIN     = 800
FRAC_MARGEN  = 0.08
VENTANA_SUAVIZADO = 5
FACTOR_UMBRAL = 1.05
V_MAX_TEXTURA = 70

GABOR_PARAMS = [
    {"freq": 1/12, "theta": 0},
    {"freq": 1/12, "theta": np.pi/4},
    {"freq": 1/16, "theta": 0},
    {"freq": 1/16, "theta": np.pi/4},
]
kernels = [cv2.getGaborKernel((31, 31), 4.0, p["theta"], 1.0/p["freq"], 0.5, 0, ktype=cv2.CV_32F)
           for p in GABOR_PARAMS]

RANGO_TINTA_MORADA = {"h_min": 138, "h_max": 163, "s_min": 30}
RANGO_TINTA_AZUL   = {"h_min": 100, "h_max": 135, "s_min": 30}
RANGO_CINTA_ROJA = {
    "h_banda_baja":  {"h_min": 0,   "h_max": 8},
    "h_banda_alta":  {"h_min": 170, "h_max": 179},
    "s_min": 100,
    "v_min": 100,
}
V_MIN_PISO = 140
S_MAX_PISO = 20
S_MIN_TINTA_PURA = 60

ESCALA_SALIDA = 1.0

def mascara_correccion_post(roi_bgr):
    roi_hsv = cv2.cvtColor(roi_bgr, cv2.COLOR_BGR2HSV)
    H, S, V = cv2.split(roi_hsv)
    mask_morada = (H >= RANGO_TINTA_MORADA["h_min"]) & (H <= RANGO_TINTA_MORADA["h_max"]) & (S >= S_MIN_TINTA_PURA)
    mask_azul   = (H >= RANGO_TINTA_AZUL["h_min"])   & (H <= RANGO_TINTA_AZUL["h_max"])   & (S >= S_MIN_TINTA_PURA)
    mask_piso   = (V > V_MIN_PISO) & (S < S_MAX_PISO)
    mask_roja_baja = (H >= RANGO_CINTA_ROJA["h_banda_baja"]["h_min"]) & (H <= RANGO_CINTA_ROJA["h_banda_baja"]["h_max"]) & (S >= RANGO_CINTA_ROJA["s_min"]) & (V >= RANGO_CINTA_ROJA["v_min"])
    mask_roja_alta = (H >= RANGO_CINTA_ROJA["h_banda_alta"]["h_min"]) & (H <= RANGO_CINTA_ROJA["h_banda_alta"]["h_max"]) & (S >= RANGO_CINTA_ROJA["s_min"]) & (V >= RANGO_CINTA_ROJA["v_min"])
    mask_roja = mask_roja_baja | mask_roja_alta
    mask = (mask_morada | mask_azul | mask_roja | mask_piso).astype(np.uint8) * 255
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3)))
    return mask

def varianza_local(gris, kernel_size):
    f = gris.astype(np.float32)
    return np.clip(cv2.blur(f**2, (kernel_size, kernel_size)) - cv2.blur(f, (kernel_size, kernel_size))**2, 0, None)

def binarizar_animal_oscuro(gris, combinado, factor_umbral=FACTOR_UMBRAL):
    t_otsu, _ = cv2.threshold(combinado, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    t_ajustado = float(np.clip(t_otsu * factor_umbral, 0, 255))
    _, mask_a = cv2.threshold(combinado, t_ajustado, 255, cv2.THRESH_BINARY)
    _, mask_b = cv2.threshold(combinado, t_ajustado, 255, cv2.THRESH_BINARY_INV)
    brillo_a = gris[mask_a == 255].mean() if np.any(mask_a == 255) else 255
    brillo_b = gris[mask_b == 255].mean() if np.any(mask_b == 255) else 255
    return mask_a if brillo_a < brillo_b else mask_b

def componente_mayor(mask_bin):
    n_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(mask_bin, connectivity=8)
    if n_labels <= 1:
        return mask_bin.copy()
    idx_mayor = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
    return np.where(labels == idx_mayor, 255, 0).astype(np.uint8)

def centro_momentos(mask_c):
    M = cv2.moments(mask_c, binaryImage=True)
    if M["m00"] == 0:
        return None
    return M["m10"] / M["m00"], M["m01"] / M["m00"]

def snap_con_margen(mask_c, cx, cy, contorno, frac=FRAC_MARGEN):
    bx, by, bw, bh = cv2.boundingRect(contorno)
    mask_segura = mask_c
    for frac_actual in [frac, frac * 0.6, frac * 0.3, 0]:
        k = max(1, int(min(bw, bh) * frac_actual))
        if k <= 1:
            mask_segura = mask_c
        else:
            k += (k % 2 == 0)
            kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))
            mask_segura = cv2.erode(mask_c, kernel)
            mask_segura = componente_mayor(mask_segura)
        if np.sum(mask_segura == 255) > 0:
            break
    cx_i, cy_i = int(round(cx)), int(round(cy))
    if 0 <= cy_i < mask_segura.shape[0] and 0 <= cx_i < mask_segura.shape[1] and mask_segura[cy_i, cx_i] == 255:
        return cx_i, cy_i, False
    ys, xs = np.where(mask_segura == 255)
    dist2 = (xs - cx) ** 2 + (ys - cy) ** 2
    idx = np.argmin(dist2)
    return int(xs[idx]), int(ys[idx]), True

def segmentar_y_centrar_final(roi_bgr, mascara_estructuras_roi):
    mask_correccion = mascara_correccion_post(roi_bgr)
    mask_neutralizar = cv2.bitwise_or(mascara_estructuras_roi, mask_correccion)
    roi_limpio = cv2.inpaint(roi_bgr, mask_neutralizar, 7, cv2.INPAINT_TELEA) if np.any(mask_neutralizar == 255) else roi_bgr

    v_original = cv2.cvtColor(roi_limpio, cv2.COLOR_BGR2HSV)[:, :, 2]

    suave  = cv2.GaussianBlur(roi_limpio, (KERNEL_GAUSS, KERNEL_GAUSS), 0)
    gris   = cv2.cvtColor(suave, cv2.COLOR_BGR2GRAY)
    gris   = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8)).apply(gris)
    gris_f = gris.astype(np.float32)

    suma = np.zeros_like(gris_f)
    for k in kernels:
        suma += np.abs(cv2.filter2D(gris_f, cv2.CV_32F, k))
    gabor_inv = cv2.bitwise_not(cv2.normalize(suma / len(kernels), None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8))
    var_norm  = cv2.normalize(varianza_local(gris, KERNEL_VAR), None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    combinado = cv2.addWeighted(gabor_inv, 0.6, var_norm, 0.4, 0)

    mask = binarizar_animal_oscuro(gris, combinado)

    mask_oscuro = (v_original < V_MAX_TEXTURA).astype(np.uint8) * 255
    mask = cv2.bitwise_and(mask, mask_oscuro)

    mask = cv2.bitwise_and(mask, cv2.bitwise_not(mascara_estructuras_roi))

    w_roi, h_roi = roi_bgr.shape[1], roi_bgr.shape[0]
    MORPH_CLOSE = max(7, int(min(w_roi, h_roi) * 0.06)); MORPH_CLOSE += (MORPH_CLOSE % 2 == 0)
    MORPH_OPEN  = max(5, int(min(w_roi, h_roi) * 0.06)); MORPH_OPEN  += (MORPH_OPEN  % 2 == 0)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (MORPH_CLOSE, MORPH_CLOSE)))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (MORPH_OPEN, MORPH_OPEN)))

    mask_flood = mask.copy()
    hf, wf = mask_flood.shape
    flood_mask = np.zeros((hf + 2, wf + 2), np.uint8)
    cv2.floodFill(mask_flood, flood_mask, (0, 0), 255)
    mask_rellena = mask | cv2.bitwise_not(mask_flood)
    if np.sum(mask_rellena == 255) / (hf * wf) < 0.95:
        mask = mask_rellena

    contornos, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    mask_c_vacia = np.zeros(mask.shape, dtype=np.uint8)
    if not contornos:
        return mask_c_vacia, None

    c_mayor = max(contornos, key=cv2.contourArea)
    if cv2.contourArea(c_mayor) < AREA_MIN:
        return mask_c_vacia, None

    mask_c = np.zeros(mask.shape, dtype=np.uint8)
    cv2.drawContours(mask_c, [c_mayor], -1, 255, cv2.FILLED)
    mask_c = componente_mayor(mask_c)

    centro = centro_momentos(mask_c)
    if centro is None:
        return mask_c, None

    cx_m, cy_m, hubo_snap = snap_con_margen(mask_c, centro[0], centro[1], c_mayor)
    return mask_c, (cx_m, cy_m, hubo_snap)

def calcular_largo_px(mask_c):
    """Largo cabeza-cola = distancia recta MÁXIMA entre dos puntos del contorno del animal.
    Baja cuando el animal se arquea/contrae, sube cuando se estira — no mide el
    contorno, mide la distancia recta entre los dos extremos más separados."""
    contornos, _ = cv2.findContours(mask_c, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contornos:
        return None, None, None
    c = max(contornos, key=cv2.contourArea)
    hull = cv2.convexHull(c).reshape(-1, 2)
    if len(hull) < 2:
        return None, None, None
    max_dist = 0.0
    p1 = p2 = None
    for i in range(len(hull)):
        for j in range(i + 1, len(hull)):
            d = float(np.hypot(hull[i][0] - hull[j][0], hull[i][1] - hull[j][1]))
            if d > max_dist:
                max_dist, p1, p2 = d, hull[i], hull[j]
    return max_dist, p1, p2

#PROCESAMIENTO
nombre_video = list(video.keys())[0]
w, h = roi_coords_final["w"], roi_coords_final["h"]

cap = cv2.VideoCapture(nombre_video)
fps   = cap.get(cv2.CAP_PROP_FPS)
ancho = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
alto  = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

roi_x = max(0, min(roi_coords_final["x"], ancho - w))
roi_y = max(0, min(roi_coords_final["y"], alto - h))

historial_cx = deque(maxlen=VENTANA_SUAVIZADO)

ancho_salida = int(ancho * 2 * ESCALA_SALIDA)
alto_salida  = int(alto * ESCALA_SALIDA)

nombre_out_video = nombre_video.replace(".mp4", "_opcion3_exclusion_auto.mp4")
out_video = cv2.VideoWriter(nombre_out_video, cv2.VideoWriter_fourcc(*'mp4v'), fps, (ancho_salida, alto_salida), isColor=True)

frame_idx = 0
largos_px = []

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_idx += 1
    if frame_idx % 50 == 0:
        print(f"  Frame {frame_idx}/{total_frames}...  ROI en x={roi_x} (y fijo en {roi_y})")

    roi_x = max(0, min(roi_x, ancho - w))
    roi_y = max(0, min(roi_y, alto - h))

    roi_bgr = frame[roi_y:roi_y + h, roi_x:roi_x + w]
    mascara_estructuras_roi = MASCARA_ESTRUCTURAS[roi_y:roi_y + h, roi_x:roi_x + w]

    mask_c, resultado = segmentar_y_centrar_final(roi_bgr, mascara_estructuras_roi)

    largo_px, p1, p2 = calcular_largo_px(mask_c)
    largos_px.append(largo_px)

    panel_izq = frame.copy()
    cv2.rectangle(panel_izq, (roi_x, roi_y), (roi_x + w, roi_y + h), (0, 255, 0), 1)

    panel_mask = np.zeros_like(frame)
    mask_color = cv2.cvtColor(mask_c, cv2.COLOR_GRAY2BGR)
    panel_mask[roi_y:roi_y + h, roi_x:roi_x + w] = mask_color
    cv2.rectangle(panel_mask, (roi_x, roi_y), (roi_x + w, roi_y + h), (0, 255, 0), 1)

    if p1 is not None and p2 is not None:
        p1_full = (roi_x + int(p1[0]), roi_y + int(p1[1]))
        p2_full = (roi_x + int(p2[0]), roi_y + int(p2[1]))
        cv2.line(panel_izq, p1_full, p2_full, (255, 0, 0), 1)
        cv2.line(panel_mask, p1_full, p2_full, (255, 0, 0), 1)
        cv2.putText(panel_mask, f"{largo_px:.1f}px", (p1_full[0], max(0, p1_full[1] - 8)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 0, 0), 1)

    cx_final = cy_final = None
    if resultado is not None:
        cx_roi, cy_roi, hubo_snap = resultado
        cx_final = roi_x + cx_roi
        cy_final = roi_y + cy_roi

        color = (0, 165, 255) if hubo_snap else (0, 0, 255)
        cv2.circle(panel_izq, (cx_final, cy_final), 6, color, -1)
        cv2.circle(panel_mask, (cx_final, cy_final), 6, color, -1)

        historial_cx.append(cx_final)
        cx_suavizado = int(round(sum(historial_cx) / len(historial_cx)))
        roi_x = max(0, min(cx_suavizado - w // 2, ancho - w))

    frame_comparado = cv2.hconcat([panel_izq, panel_mask])
    if ESCALA_SALIDA != 1.0:
        frame_comparado = cv2.resize(frame_comparado, (ancho_salida, alto_salida))

    out_video.write(frame_comparado)

cap.release()
out_video.release()

print(f"Listo — video guardado como {nombre_out_video}")

#CSV LARGO
nombre_csv = nombre_video.replace(".mp4", "_largos_px.csv")
with open(nombre_csv, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["frame", "largo_px"])
    for i, l in enumerate(largos_px, start=1):
        writer.writerow([i, "" if l is None else round(l, 2)])
files.download(nombre_csv)

validos = [l for l in largos_px if l is not None]
if validos:
    print(f"Largo (px): min={min(validos):.1f}  max={max(validos):.1f}  promedio={sum(validos)/len(validos):.1f}"
          f" (frames con animal detectado: {len(validos)}/{len(largos_px)})")

nombre_h264 = nombre_out_video.replace(".mp4", "_h264.mp4")
subprocess.run(["ffmpeg", "-y", "-i", nombre_out_video, "-vcodec", "libx264", "-pix_fmt", "yuv420p", nombre_h264],
               check=True, capture_output=True)

with open(nombre_h264, "rb") as f:
    video_b64 = base64.b64encode(f.read()).decode()
display(HTML(f'<video width="900" controls><source src="data:video/mp4;base64,{video_b64}" type="video/mp4"></video>'))

from google.colab import files
files.download(nombre_h264)

In [ ]:
# CELDA 3 v.1
import matplotlib.pyplot as plt

frames_validos = [i+1 for i, l in enumerate(largos_px) if l is not None]
largos_validos = [l for l in largos_px if l is not None]

plt.figure(figsize=(12, 4))
plt.plot(frames_validos, largos_validos, linewidth=0.8)
plt.xlabel("Frame")
plt.ylabel("Largo (px)")
plt.title(f"Largo del animal por frame — {nombre_video}")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("grafico_largo.png", dpi=150)
plt.show()

from google.colab import files
files.download("grafico_largo.png")

In [ ]:
# CELDA 3 v.2
import numpy as np
import matplotlib.pyplot as plt

VENTANA_MEDIANA = 5

def mediana_movil(valores, ventana):
    arr = np.array([np.nan if v is None else v for v in valores], dtype=float)
    suavizado = arr.copy()
    mitad = ventana // 2
    for i in range(len(arr)):
        ini, fin = max(0, i - mitad), min(len(arr), i + mitad + 1)
        vecinos = arr[ini:fin]
        vecinos = vecinos[~np.isnan(vecinos)]
        if len(vecinos) > 0:
            suavizado[i] = np.median(vecinos)
    return suavizado

largos_suavizados = mediana_movil(largos_px, VENTANA_MEDIANA)

plt.figure(figsize=(12, 4))
plt.plot(largos_px, linewidth=0.6, alpha=0.4, label="crudo")
plt.plot(largos_suavizados, linewidth=1.0, label=f"mediana móvil ({VENTANA_MEDIANA} frames)")
plt.xlabel("Frame")
plt.ylabel("Largo (px)")
plt.title(f"Largo del animal por frame — {nombre_video}")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("grafico_largo_suavizado.png", dpi=150)
plt.show()

from google.colab import files
files.download("grafico_largo_suavizado.png")

import csv
nombre_csv_suave = nombre_video.replace(".mp4", "_largos_px_suavizado.csv")
with open(nombre_csv_suave, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["frame", "largo_px_crudo", "largo_px_suavizado"])
    for i, (crudo, suave) in enumerate(zip(largos_px, largos_suavizados), start=1):
        writer.writerow([i, "" if crudo is None else round(crudo, 2), round(suave, 2)])
files.download(nombre_csv_suave)

In [ ]:
# CELDA 3 v.3
import numpy as np
import matplotlib.pyplot as plt
import csv

VENTANA_MEDIANA = 7   # subida de 5 a 7 para tapar el parpadeo más largo (~3355-3361)

def mediana_movil(valores, ventana):
    arr = np.array([np.nan if v is None else v for v in valores], dtype=float)
    suavizado = arr.copy()
    mitad = ventana // 2
    minimo_validos = mitad + 1   # necesita mayoría de vecinos válidos, si no deja NaN (no inventa en huecos largos)
    for i in range(len(arr)):
        ini, fin = max(0, i - mitad), min(len(arr), i + mitad + 1)
        vecinos = arr[ini:fin]
        vecinos = vecinos[~np.isnan(vecinos)]
        if len(vecinos) >= minimo_validos:
            suavizado[i] = np.median(vecinos)
        else:
            suavizado[i] = np.nan
    return suavizado

largos_suavizados = mediana_movil(largos_px, VENTANA_MEDIANA)

plt.figure(figsize=(12, 4))
plt.plot(largos_px, linewidth=0.6, alpha=0.4, label="crudo")
plt.plot(largos_suavizados, linewidth=1.0, label=f"mediana móvil ({VENTANA_MEDIANA} frames, sin inventar en huecos largos)")
plt.xlabel("Frame")
plt.ylabel("Largo (px)")
plt.title(f"Largo del animal por frame — {nombre_video}")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("grafico_largo_suavizado_v2.png", dpi=150)
plt.show()

from google.colab import files
files.download("grafico_largo_suavizado_v2.png")

nombre_csv_suave = nombre_video.replace(".mp4", "_largos_px_suavizado_v2.csv")
with open(nombre_csv_suave, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["frame", "largo_px_crudo", "largo_px_suavizado"])
    for i, (crudo, suave) in enumerate(zip(largos_px, largos_suavizados), start=1):
        writer.writerow([i, "" if crudo is None else round(crudo, 2), "" if np.isnan(suave) else round(suave, 2)])
files.download(nombre_csv_suave)

n_huecos = int(np.isnan(largos_suavizados).sum())
print(f"✅ Frames sin dato confiable (hueco real, sin animal detectado de forma sostenida): {n_huecos}/{len(largos_suavizados)}")

In [ ]:
# CELDA 3 v.4
import numpy as np
import matplotlib.pyplot as plt
import csv

VENTANA_MEDIANA = 7

def mediana_movil(valores, ventana):
    arr = np.array([np.nan if v is None else v for v in valores], dtype=float)
    suavizado = arr.copy()
    mitad = ventana // 2
    minimo_validos = mitad + 1
    for i in range(len(arr)):
        ini, fin = max(0, i - mitad), min(len(arr), i + mitad + 1)
        vecinos = arr[ini:fin]
        vecinos = vecinos[~np.isnan(vecinos)]
        if len(vecinos) >= minimo_validos:
            suavizado[i] = np.median(vecinos)
        else:
            suavizado[i] = np.nan
    return suavizado

largos_suavizados = mediana_movil(largos_px, VENTANA_MEDIANA)

# ── eje X en segundos, usando el fps del video (ya calculado en Celda B) ──
segundos = np.arange(1, len(largos_px) + 1) / fps

plt.figure(figsize=(12, 4))
plt.plot(segundos, largos_px, linewidth=0.6, alpha=0.4, label="crudo")
plt.plot(segundos, largos_suavizados, linewidth=1.0, label=f"mediana móvil ({VENTANA_MEDIANA} frames)")
plt.xlabel("Tiempo (s)")
plt.ylabel("Largo (px)")
plt.title(f"Largo del animal por segundo — {nombre_video}")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("grafico_largo_suavizado_seg.png", dpi=150)
plt.show()

from google.colab import files
files.download("grafico_largo_suavizado_seg.png")

nombre_csv_suave = nombre_video.replace(".mp4", "_largos_px_suavizado_seg.csv")
with open(nombre_csv_suave, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["frame", "segundo", "largo_px_crudo", "largo_px_suavizado"])
    for i, (seg, crudo, suave) in enumerate(zip(segundos, largos_px, largos_suavizados), start=1):
        writer.writerow([i, round(seg, 2), "" if crudo is None else round(crudo, 2), "" if np.isnan(suave) else round(suave, 2)])
files.download(nombre_csv_suave)

n_huecos = int(np.isnan(largos_suavizados).sum())
print(f"Frames sin dato confiable: {n_huecos}/{len(largos_suavizados)}")
print(f"Video de {len(largos_px)} frames a {fps:.2f} fps → {segundos[-1]:.1f} segundos")